<a href="https://colab.research.google.com/github/hiten4/Day37/blob/main/Day37.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 07 NLP Assignment - ShopSense

TF-IDF from scratch, similarity search, and evaluation.

In [9]:

import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from math import log
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
reviews = pd.read_csv('shopsense_reviews.csv')

In [11]:

if 'review_text' not in reviews.columns:
    raise ValueError("review_text column missing")

reviews['review_text'] = reviews['review_text'].fillna("").astype(str)

In [12]:
# Preprocess
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

reviews['tokens'] = reviews['review_text'].apply(preprocess)

In [13]:
# TF
def compute_tf(tokens):
    tf = Counter(tokens)
    total = len(tokens)
    return {w: c/total for w, c in tf.items()}

reviews['tf'] = reviews['tokens'].apply(compute_tf)

# IDF
def compute_idf(docs):
    N = len(docs)
    df = defaultdict(int)
    for doc in docs:
        for word in set(doc):
            df[word] += 1
    return {w: log(N/(1+f)) for w, f in df.items()}

idf = compute_idf(reviews['tokens'])

# TF-IDF
def compute_tfidf(tf, idf):
    return {w: tf[w]*idf.get(w,0) for w in tf}

reviews['tfidf'] = reviews['tf'].apply(lambda x: compute_tfidf(x, idf))

In [14]:
# Vectorization
vocab = list(idf.keys())
vocab_index = {w:i for i,w in enumerate(vocab)}

def to_vec(d):
    v = np.zeros(len(vocab))
    for w,val in d.items():
        if w in vocab_index:
            v[vocab_index[w]] = val
    return v

tfidf_matrix = np.array([to_vec(d) for d in reviews['tfidf']])

In [15]:
# Query
query = "wireless earbuds battery life poor"
qt = preprocess(query)
qtf = compute_tf(qt)
qtfidf = compute_tfidf(qtf, idf)
qvec = to_vec(qtfidf).reshape(1,-1)

sim = cosine_similarity(qvec, tfidf_matrix)[0]
top5 = np.argsort(sim)[-5:][::-1]

print("Top 5 Reviews:")
for i in top5:
    print(sim[i], reviews.iloc[i]['review_text'])

# Sklearn comparison
vec = TfidfVectorizer()
sk = vec.fit_transform(reviews['review_text']).toarray()

min_dim = min(tfidf_matrix.shape[1], sk.shape[1])
l2 = np.linalg.norm(tfidf_matrix[:,:min_dim] - sk[:,:min_dim], axis=1)
print("Avg L2:", np.mean(l2))

# Electronics top word
elec = reviews[reviews['category']=='Electronics']
scores = defaultdict(list)
for d in elec['tfidf']:
    for w,s in d.items():
        scores[w].append(s)

avg = {w:np.mean(v) for w,v in scores.items()}
print("Top word:", max(avg, key=avg.get))


Top 5 Reviews:
0.30462896824966756 Superb! The earbuds is exactly as described. Very happy with this purchase.
0.30462896824966756 Superb! The earbuds is exactly as described. Very happy with this purchase.
0.30462896824966756 Superb! The earbuds is exactly as described. Very happy with this purchase.
0.30462896824966756 Superb! The earbuds is exactly as described. Very happy with this purchase.
0.30462896824966756 Superb! The earbuds is exactly as described. Very happy with this purchase.
Avg L2: 1.095914418495597
Top word: na
